# 🛣️ Road Guard AI — YOLOv8 Training Notebook
**Train pothole detection model from your local dataset and export `best.pt`**

### Before you start:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload your dataset zip in **Cell 3** (instructions inside)
3. Run all cells top to bottom

---

## Cell 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## Cell 2 — Install Dependencies

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
import yaml, os, shutil
from pathlib import Path
print("✅ Ultralytics installed")

## Cell 3 — Upload & Extract Your Dataset

Your local dataset should be a ZIP with this structure:
```
dataset/
├── train/
│   ├── images/   ← .jpg / .png
│   └── labels/   ← .txt (YOLO format)
├── valid/
│   ├── images/
│   └── labels/
└── data.yaml
```

**If your dataset is NOT in YOLO format** (e.g. COCO JSON), set `CONVERT_FROM_COCO = True` below.

Upload your ZIP using the Colab file panel (left sidebar → Files → Upload), then set the filename below.

In [ ]:
# ─── CONFIG — edit these ───────────────────────────────────────────
DATASET_ZIP      = "dataset.zip"      # name of the zip you uploaded
DATASET_DIR      = "/content/dataset" # where it will be extracted
CONVERT_FROM_COCO = False             # set True if labels are COCO JSON
# ───────────────────────────────────────────────────────────────────

import zipfile

if not os.path.exists(DATASET_ZIP):
    # Try Google Drive mount as fallback
    print("ZIP not found in /content — trying Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    # Update this path to wherever your zip is in Drive
    DATASET_ZIP = "/content/drive/MyDrive/dataset.zip"

print(f"Extracting {DATASET_ZIP} ...")
with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    z.extractall(DATASET_DIR)
print(f"✅ Extracted to {DATASET_DIR}")

# List what's inside
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files[:5]:
            print(f"{subindent}{f}")
        if len(files) > 5:
            print(f"{subindent}... and {len(files)-5} more")

## Cell 4 — COCO → YOLO Conversion (skip if already YOLO format)

In [ ]:
if CONVERT_FROM_COCO:
    import json

    def coco_to_yolo(json_path, images_dir, output_labels_dir):
        os.makedirs(output_labels_dir, exist_ok=True)
        with open(json_path) as f:
            coco = json.load(f)

        img_id_to_info = {img['id']: img for img in coco['images']}
        # group annotations by image
        from collections import defaultdict
        ann_by_img = defaultdict(list)
        for ann in coco['annotations']:
            ann_by_img[ann['image_id']].append(ann)

        for img_id, img_info in img_id_to_info.items():
            W, H = img_info['width'], img_info['height']
            fname = Path(img_info['file_name']).stem
            label_path = os.path.join(output_labels_dir, fname + '.txt')
            with open(label_path, 'w') as lf:
                for ann in ann_by_img[img_id]:
                    x, y, w, h = ann['bbox']  # COCO: top-left x,y,w,h
                    cx = (x + w/2) / W
                    cy = (y + h/2) / H
                    nw = w / W
                    nh = h / H
                    cat = ann['category_id'] - 1  # 0-indexed
                    lf.write(f"{cat} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")
        print(f"✅ Converted {len(img_id_to_info)} images → {output_labels_dir}")

    # Run for train and valid splits
    # Update these paths to match your zip structure
    coco_to_yolo(
        json_path=f"{DATASET_DIR}/train/_annotations.coco.json",
        images_dir=f"{DATASET_DIR}/train/images",
        output_labels_dir=f"{DATASET_DIR}/train/labels"
    )
    coco_to_yolo(
        json_path=f"{DATASET_DIR}/valid/_annotations.coco.json",
        images_dir=f"{DATASET_DIR}/valid/images",
        output_labels_dir=f"{DATASET_DIR}/valid/labels"
    )
else:
    print("⏭️  Skipping COCO conversion (already YOLO format)")

## Cell 5 — Create / Verify data.yaml

In [ ]:
YAML_PATH = f"{DATASET_DIR}/data.yaml"

# Check if data.yaml already exists in the zip
if os.path.exists(YAML_PATH):
    with open(YAML_PATH) as f:
        content = f.read()
    print("Found existing data.yaml:")
    print(content)

    # Fix paths to be absolute (common issue when moving datasets)
    data = yaml.safe_load(content)
    data['path'] = DATASET_DIR
    data['train'] = 'train/images'
    data['val']   = 'valid/images'
    with open(YAML_PATH, 'w') as f:
        yaml.dump(data, f)
    print("\n✅ Paths updated to absolute")

else:
    # Create fresh data.yaml
    # ─── Edit class names if needed ───
    CLASS_NAMES = ["pothole"]  # add more if your dataset has multiple classes
    # ──────────────────────────────────

    data = {
        'path'  : DATASET_DIR,
        'train' : 'train/images',
        'val'   : 'valid/images',
        'nc'    : len(CLASS_NAMES),
        'names' : CLASS_NAMES
    }
    with open(YAML_PATH, 'w') as f:
        yaml.dump(data, f)
    print(f"✅ Created data.yaml with classes: {CLASS_NAMES}")

# Final check — count images
train_imgs = len(list(Path(f"{DATASET_DIR}/train/images").glob("*.*")))
valid_imgs = len(list(Path(f"{DATASET_DIR}/valid/images").glob("*.*")))
print(f"\n📊 Train images : {train_imgs}")
print(f"📊 Valid images : {valid_imgs}")

## Cell 6 — Train YOLOv8s

Using `yolov8s` (small) — good balance of accuracy vs speed for pothole detection.
T4 has 15GB VRAM so no memory constraints here.

In [ ]:
# ─── TRAINING CONFIG ───────────────────────────────────────────────
MODEL_SIZE  = "yolov8s.pt"   # nano=yolov8n, small=yolov8s, medium=yolov8m
EPOCHS      = 80             # increase to 100-150 if you have time
BATCH_SIZE  = 16             # T4 can handle 16 easily
IMG_SIZE    = 640            # standard YOLO input size
PROJECT     = "road_guard"   # output folder name
RUN_NAME    = "pothole_v1"   # run subfolder name
# ───────────────────────────────────────────────────────────────────

model = YOLO(MODEL_SIZE)

results = model.train(
    data      = YAML_PATH,
    epochs    = EPOCHS,
    batch     = BATCH_SIZE,
    imgsz     = IMG_SIZE,
    project   = PROJECT,
    name      = RUN_NAME,
    device    = 0,           # GPU
    patience  = 20,          # early stopping if no improvement for 20 epochs
    save      = True,
    plots     = True,        # saves confusion matrix, PR curve etc.
    verbose   = True,
    # Augmentation — good for road datasets
    flipud    = 0.0,         # no vertical flip (potholes are always on ground)
    fliplr    = 0.5,
    mosaic    = 1.0,
    degrees   = 5.0,         # slight rotation for bike camera tilt
    translate = 0.1,
    scale     = 0.3,
    hsv_h     = 0.015,
    hsv_s     = 0.5,
    hsv_v     = 0.3,         # brightness variation for different lighting
)

print("\n✅ Training complete!")
print(f"Best model saved at: {PROJECT}/{RUN_NAME}/weights/best.pt")

## Cell 7 — Evaluate on Validation Set

In [ ]:
best_model_path = f"{PROJECT}/{RUN_NAME}/weights/best.pt"
model_eval = YOLO(best_model_path)

metrics = model_eval.val(data=YAML_PATH, imgsz=IMG_SIZE)

print("\n📊 Validation Results:")
print(f"  mAP50      : {metrics.box.map50:.4f}")
print(f"  mAP50-95   : {metrics.box.map:.4f}")
print(f"  Precision  : {metrics.box.mp:.4f}")
print(f"  Recall     : {metrics.box.mr:.4f}")

# Show training plots
from IPython.display import Image, display
plots_dir = f"{PROJECT}/{RUN_NAME}"
for plot in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    path = f"{plots_dir}/{plot}"
    if os.path.exists(path):
        print(f"\n{plot}:")
        display(Image(path))

## Cell 8 — Test Inference on a Sample Image

In [ ]:
import glob
from IPython.display import Image, display

# Grab a random validation image
sample_imgs = glob.glob(f"{DATASET_DIR}/valid/images/*.*")[:3]

model_infer = YOLO(best_model_path)

for img_path in sample_imgs:
    results = model_infer.predict(
        source     = img_path,
        conf       = 0.25,
        save       = True,
        project    = "inference_test",
        name       = "samples",
        exist_ok   = True
    )
    for r in results:
        print(f"Image: {img_path}")
        print(f"  Detections: {len(r.boxes)}")
        for box in r.boxes:
            conf = float(box.conf)
            cls  = int(box.cls)
            xyxy = box.xyxy[0].tolist()
            print(f"  → class={cls}, conf={conf:.2f}, bbox={[round(x,1) for x in xyxy]}")

# Show saved inference results
result_imgs = glob.glob("inference_test/samples/*.*")
for img in result_imgs[:3]:
    display(Image(img))

## Cell 9 — Export best.pt + Download

This downloads `best.pt` directly to your machine.

In [ ]:
import shutil
from google.colab import files

best_pt = f"{PROJECT}/{RUN_NAME}/weights/best.pt"
output_name = "road_guard_pothole_best.pt"

shutil.copy(best_pt, output_name)
print(f"📦 Copied to {output_name}")

# Download
files.download(output_name)
print("✅ Download started!")

## Cell 10 — (Optional) Also Save to Google Drive

In [ ]:
# Run this if the download in Cell 9 fails or you want a backup
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_SAVE_PATH = "/content/drive/MyDrive/RoadGuardAI/"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

# Save model
shutil.copy(best_pt, f"{DRIVE_SAVE_PATH}/road_guard_pothole_best.pt")

# Save training plots too
plots_src = f"{PROJECT}/{RUN_NAME}/"
shutil.copytree(plots_src, f"{DRIVE_SAVE_PATH}/training_run/", dirs_exist_ok=True)

print(f"✅ Model + plots saved to Google Drive: {DRIVE_SAVE_PATH}")

## Cell 11 — (Optional) Export to ONNX for FastAPI deployment

In [ ]:
# ONNX export — useful if you want CPU inference on your FastAPI server
# without needing torch/ultralytics installed on the server

model_export = YOLO(best_pt)
model_export.export(format="onnx", imgsz=IMG_SIZE, simplify=True)

onnx_path = best_pt.replace(".pt", ".onnx")
shutil.copy(onnx_path, "road_guard_pothole_best.onnx")

from google.colab import files
files.download("road_guard_pothole_best.onnx")
print("✅ ONNX model downloaded")

---
## 📋 What you now have

| File | Use |
|------|-----|
| `road_guard_pothole_best.pt` | Load with `YOLO('best.pt')` in FastAPI |
| `road_guard_pothole_best.onnx` | CPU inference without torch dependency |

## Next step — FastAPI inference
```python
from ultralytics import YOLO
model = YOLO('road_guard_pothole_best.pt')

results = model.predict(source=frame, conf=0.25)
boxes = results[0].boxes  # bounding boxes
```

Pair with **Depth Anything v2** on the same frame to get pothole depth estimate.